In [19]:
%pip install -q pandas numpy scikit-learn matplotlib seaborn joblib

Note: you may need to restart the kernel to use updated packages.


In [20]:
import os
import re
import glob
import joblib
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Step 6 libraries imported successfully.")

Step 6 libraries imported successfully.


In [21]:
reports_dir = Path("reports")
reports_dir.mkdir(exist_ok=True)

report_files = list(reports_dir.glob("*.csv"))

print("Reports available:", len(report_files))

for file in report_files:
    print(file)

Reports available: 14
reports\best_model_test_metrics.csv
reports\contract_churn.csv
reports\customer_risk_segments.csv
reports\customer_segment_analysis.csv
reports\eda_summary.csv
reports\feature_importance.csv
reports\high_value_churned_customers.csv
reports\model_comparison_cv.csv
reports\optimized_model_metrics.csv
reports\overall_churn.csv
reports\payment_method_churn.csv
reports\retention_business_scenario.csv
reports\risk_segment_summary.csv
reports\sample_churn_predictions.csv


#### Convert reports into searchable documents

In [22]:
documents = []

for file in report_files:
    try:
        report_df = pd.read_csv(file)

        if report_df.empty:
            continue

        report_text = report_df.to_string(
            index=False,
            max_rows=100
        )

        documents.append({
            "source": file.name,
            "text": (
                f"Report name: {file.stem}\n"
                f"Report data:\n{report_text}"
            )
        })

    except Exception as e:
        print(f"Could not read {file.name}: {e}")

documents_df = pd.DataFrame(documents)

print("Searchable documents:", len(documents_df))
documents_df.head()

Searchable documents: 14


,source,text
0,best_model_test_metrics.csv,Report name: best_model_test_metrics\nReport d...
1,contract_churn.csv,Report name: contract_churn\nReport data:\n ...
2,customer_risk_segments.csv,Report name: customer_risk_segments\nReport da...
3,customer_segment_analysis.csv,Report name: customer_segment_analysis\nReport...
4,eda_summary.csv,Report name: eda_summary\nReport data:\n ...


### Build the retrieval engine

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    analyzer="word",
    sublinear_tf=True
)

document_matrix = vectorizer.fit_transform(
    documents_df["text"].fillna("")
)

print("Updated retrieval engine:", document_matrix.shape)

Updated retrieval engine: (14, 2733)


#### Creating a Search Function

In [24]:
def retrieve_relevant_docs(question, top_k=3):
    if not question or not question.strip():
        return pd.DataFrame(
            columns=["source", "similarity", "text"]
        )

    query_vector = vectorizer.transform([question])

    if query_vector.nnz == 0:
        return pd.DataFrame(
            columns=["source", "similarity", "text"]
        )

    similarities = cosine_similarity(
        query_vector, document_matrix
    ).flatten()

    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = documents_df.iloc[top_indices].copy()
    results["similarity"] = similarities[top_indices]

    return results[
        ["source", "similarity", "text"]
    ].reset_index(drop=True)

In [25]:
question = "Which customer segments have high churn?"

results = retrieve_relevant_docs(question, top_k=3)

for _, row in results.iterrows():
    print(f"\nSource: {row['source']}")
    print(f"Similarity: {row['similarity']:.3f}")
    print(row["text"][:1500])
    print("-" * 60)


Source: risk_segment_summary.csv
Similarity: 0.101
Report name: risk_segment_summary
Report data:
Risk_Segment  customers  avg_predicted_risk  actual_churn_rate  avg_monthly_charges  avg_predicted_risk_pct  actual_churn_rate_pct
        High        104            0.784156           0.817308            81.539423                   78.42                  81.73
         Low        989            0.128358           0.123357            58.579474                   12.84                  12.34
      Medium        316            0.531453           0.528481            75.588608                   53.15                  52.85
------------------------------------------------------------

Source: customer_risk_segments.csv
Similarity: 0.090
Report name: customer_risk_segments
Report data:
 Unnamed: 0 gender  SeniorCitizen Partner Dependents  tenure PhoneService MultipleLines InternetService      OnlineSecurity        OnlineBackup    DeviceProtection         TechSupport         StreamingTV     Strea

### Add a basic evidence-based assistant

In [26]:
def customer_insights_assistant(question, top_k=3):
    results = retrieve_relevant_docs(
        question,
        top_k=top_k
    )

    if results.empty:
        return "Please provide a question about the project reports."

    # Avoid presenting irrelevant documents as evidence.
    results = results[results["similarity"] > 0]

    if results.empty:
        return (
            "I could not find relevant evidence in the available "
            "reports. Try a more specific question."
        )

    answer = [
        "CUSTOMER INTELLIGENCE ASSISTANT",
        f"Question: {question}",
        "",
        "Relevant evidence:"
    ]

    for _, row in results.iterrows():
        answer.append(f"\nSource: {row['source']}")
        answer.append(row["text"][:2000])

    return "\n".join(answer)

In [27]:
questions = [
    "Which contract types have the highest churn?",
    "How did the churn prediction models perform?",
    "Which customer risk segments need attention?",
    "What retention business scenario was estimated?"
]

for question in questions:
    print("\n" + "=" * 70)
    print(customer_insights_assistant(question))


CUSTOMER INTELLIGENCE ASSISTANT
Question: Which contract types have the highest churn?

Relevant evidence:

Source: contract_churn.csv
Report name: contract_churn
Report data:
      Contract  total_customers  churned_customers  churn_rate_pct
Month-to-month             3875               1655           42.71
      One year             1473                166           11.27
      Two year             1695                 48            2.83

Source: customer_segment_analysis.csv
Report name: customer_segment_analysis
Report data:
                  segment  customers  churn_rate  avg_monthly_charges         feature  churn_rate_pct
           Month-to-month       3875    0.427097            66.398490        Contract       42.709677
                 One year       1473    0.112695            65.048608        Contract       11.269518
                 Two year       1695    0.028319            60.770413        Contract        2.831858
                      DSL       2421    0.189591        

#### Loading the optimized model

In [30]:
import joblib
import pandas as pd
import numpy as np
from pathlib import Path

model_path = Path("models/customer_churn_optimized_pipeline.joblib")
config_path = Path("models/customer_churn_config.joblib")

if not model_path.exists():
    raise FileNotFoundError(
        f"Model not found at {model_path.resolve()}. "
        "Check your project folder."
    )

model = joblib.load(model_path)

print("Model loaded successfully!")
print(type(model))

Model loaded successfully!
<class 'sklearn.pipeline.Pipeline'>


#### Create a customer prediction function

In [31]:
feature_columns = list(
    model.named_steps["preprocessor"].feature_names_in_
)

print("Number of input features:", len(feature_columns))
print(feature_columns)

Number of input features: 19
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']


In [33]:
data_path = Path("data/raw/Telco-Customer-Churn.csv")

if not data_path.exists():
    raise FileNotFoundError(
        f"Dataset not found at {data_path.resolve()}. "
        "Update data_path to the actual CSV location."
    )

df = pd.read_csv(data_path)

df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"], errors="coerce"
)

df.loc[
    df["TotalCharges"].isna() & (df["tenure"] == 0),
    "TotalCharges"
] = 0

df = df.drop_duplicates(subset=["customerID"])

# Match the features used during training.
X = df.drop(
    columns=["Churn", "customerID"],
    errors="ignore"
)

feature_columns = list(X.columns)

sample_customer = X.iloc[0].to_dict()

print("Feature count:", len(feature_columns))
print("Sample customer prepared.")

Feature count: 19
Sample customer prepared.


In [34]:
if config_path.exists():
    config = joblib.load(config_path)
    print("Saved configuration:", config)
else:
    config = {}

threshold = float(
    config.get("best_threshold", 0.5)
)

print("Classification threshold:", threshold)

Saved configuration: {'model_name': 'Gradient Boosting', 'decision_threshold': 0.14077933061020406, 'risk_thresholds': {'low_upper': 0.4, 'medium_upper': 0.7}, 'positive_class': 1, 'positive_class_meaning': 'Churn'}
Classification threshold: 0.5


#### Create and test the prediction function

In [35]:
def predict_customer_churn(customer_data):
    customer_df = pd.DataFrame([customer_data])

    missing = [
        col for col in feature_columns
        if col not in customer_df.columns
    ]

    if missing:
        raise ValueError(f"Missing features: {missing}")

    customer_df = customer_df[feature_columns]

    probability = float(
        model.predict_proba(customer_df)[0, 1]
    )

    predicted_churn = int(probability >= threshold)

    if probability >= 0.70:
        risk = "High"
        action = "Prioritize retention review"
    elif probability >= 0.40:
        risk = "Medium"
        action = "Review engagement and customer experience"
    else:
        risk = "Low"
        action = "Continue regular engagement"

    return {
        "churn_probability": round(probability, 4),
        "predicted_churn": predicted_churn,
        "risk_segment": risk,
        "suggested_action": action
    }


result = predict_customer_churn(sample_customer)
print(result)

{'churn_probability': 0.6866, 'predicted_churn': 1, 'risk_segment': 'Medium', 'suggested_action': 'Review engagement and customer experience'}


In [36]:
result = predict_customer_churn(sample_customer)

print("Customer Churn Prediction")
print("-" * 35)
print(f"Churn probability: {result['churn_probability']:.2%}")
print(f"Predicted churn: {result['predicted_churn']}")
print(f"Risk segment: {result['risk_segment']}")
print(f"Suggested action: {result['suggested_action']}")

Customer Churn Prediction
-----------------------------------
Churn probability: 68.66%
Predicted churn: 1
Risk segment: Medium
Suggested action: Review engagement and customer experience


In [37]:
# Select 10 sample customers
sample_customers = X.head(10).copy()

probabilities = model.predict_proba(sample_customers)[:, 1]

predictions = pd.DataFrame({
    "Churn_Probability": probabilities,
    "Predicted_Churn": (probabilities >= threshold).astype(int)
})

predictions["Risk_Segment"] = pd.cut(
    predictions["Churn_Probability"],
    bins=[-np.inf, 0.40, 0.70, np.inf],
    labels=["Low", "Medium", "High"],
    right=False
)

predictions["Suggested_Action"] = predictions["Risk_Segment"].map({
    "Low": "Continue regular engagement",
    "Medium": "Review customer experience",
    "High": "Prioritize retention review"
})

predictions

,Churn_Probability,Predicted_Churn,Risk_Segment,Suggested_Action
0,0.686642,1,Medium,Review customer experience
1,0.052077,0,Low,Continue regular engagement
2,0.295787,0,Low,Continue regular engagement
3,0.047179,0,Low,Continue regular engagement
4,0.682463,1,Medium,Review customer experience
5,0.851315,1,High,Prioritize retention review
6,0.427117,0,Medium,Review customer experience
7,0.191372,0,Low,Continue regular engagement
8,0.585500,1,Medium,Review customer experience
9,0.040425,0,Low,Continue regular engagement


In [38]:
Path("reports").mkdir(exist_ok=True)

predictions.to_csv(
    "reports/deployed_customer_predictions.csv",
    index=False
)

print("Predictions saved successfully!")

Predictions saved successfully!


In [39]:
risk_summary = (
    predictions.groupby("Risk_Segment", observed=False)
    .agg(
        Customers=("Predicted_Churn", "size"),
        Average_Churn_Probability=("Churn_Probability", "mean"),
        Predicted_Churn_Count=("Predicted_Churn", "sum")
    )
    .reset_index()
)

risk_summary["Average_Churn_Probability"] = (
    risk_summary["Average_Churn_Probability"].round(3)
)

risk_summary

,Risk_Segment,Customers,Average_Churn_Probability,Predicted_Churn_Count
0,Low,5,0.125,0
1,Medium,4,0.595,3
2,High,1,0.851,1
